A cute little demo showing the simplest usage of minGPT. Configured to run fine on Macbook Air in like a minute.

In [1]:
import torch
from torch.utils.data import Dataset
from torch.utils.data.dataloader import DataLoader
from mingpt.utils import set_seed
set_seed(3407)

In [2]:
import pickle

class SortDataset(Dataset):
    """ 
    Dataset for the Sort problem. E.g. for problem length 6:
    Input: 0 0 2 1 0 1 -> Output: 0 0 0 1 1 2
    Which will feed into the transformer concatenated as:
    input:  0 0 2 1 0 1 0 0 0 1 1
    output: I I I I I 0 0 0 1 1 2
    
    0 0 2 1 0 1 0 0 0 1 1
    0 2 1 0 1 0 0 0 1 1 2
    
    where I is "ignore", as the transformer is reading the input sequence
    """

    def __init__(self, split, length=6, num_digits=3):
        assert split in {'train', 'test'}
        self.split = split
        self.length = length
        self.num_digits = num_digits
    
    def __len__(self):
        return 10000 # ...
    
    def get_vocab_size(self):
        return self.num_digits
    
    def get_block_size(self):
        # the length of the sequence that will feed into transformer, 
        # containing concatenated input and the output, but -1 because
        # the transformer starts making predictions at the last input element
        return self.length * 2 - 1

    def __getitem__(self, idx):
        
        # use rejection sampling to generate an input example from the desired split
        while True:
            # generate some random integers
            inp = torch.randint(self.num_digits, size=(self.length,), dtype=torch.long)
            # half of the time let's try to boost the number of examples that 
            # have a large number of repeats, as this is what the model seems to struggle
            # with later in training, and they are kind of rate
            if torch.rand(1).item() < 0.5:
                if inp.unique().nelement() > self.length // 2:
                    # too many unqiue digits, re-sample
                    continue
            # figure out if this generated example is train or test based on its hash
            h = hash(pickle.dumps(inp.tolist()))
            inp_split = 'test' if h % 4 == 0 else 'train' # designate 25% of examples as test
            if inp_split == self.split:
                break # ok
        
        # solve the task: i.e. sort
        sol = torch.sort(inp)[0]

        # concatenate the problem specification and the solution
        cat = torch.cat((inp, sol), dim=0)

        # the inputs to the transformer will be the offset sequence
        x = cat[:-1].clone()
        y = cat[1:].clone()
        # we only want to predict at output locations, mask out the loss at the input locations
        y[:self.length-1] = -1
        return x, y


In [3]:
import random

def random_mul_instance(n):
    # n cyfr dla a i b (tu: 3)
    a = [random.randint(0,9) for _ in range(n)]
    b = [random.randint(0,9) for _ in range(n)]

    val_a = int(''.join(map(str, a)))
    val_b = int(''.join(map(str, b)))
    val_c = val_a * val_b

    # wynik zawsze 2n cyfr (dla n=3 -> 6 cyfr)
    str_c = str(val_c)
    str_c = (2*n - len(str_c)) * '0' + str_c  # pad do 2n

    c = [int(d) for d in str_c]
    return a + b + c  # długość: n + n + 2n = 4n

for _ in range(5):
    print(random_mul_instance(3))


[0, 6, 2, 5, 3, 6, 0, 3, 3, 2, 3, 2]
[4, 9, 5, 3, 7, 8, 1, 8, 7, 1, 1, 0]
[6, 4, 4, 3, 1, 4, 2, 0, 2, 2, 1, 6]
[3, 5, 9, 1, 1, 9, 0, 4, 2, 7, 2, 1]
[3, 5, 0, 1, 4, 8, 0, 5, 1, 8, 0, 0]


In [4]:
class MulDataset(Dataset):
    """
    Dataset for multiplication of two n-digit numbers.
    For n=3:
    input: a(3) + b(3)  -> 6 tokens
    output: c(6)        -> 6 tokens (zero-padded)
    concatenated: a(3) b(3) c(6) -> 12 tokens total

    x = cat[:-1] length 11
    y = cat[1:]  length 11
    mask loss on positions that correspond to reading the input digits
    """

    def __init__(self, split, n=3):
        assert split in {'train', 'test'}
        self.split = split
        self.n = n

    def __len__(self):
        return 10000

    def get_vocab_size(self):
        return 10

    def get_block_size(self):
        # total tokens = 4n, but x is cat[:-1] => 4n - 1
        return 4 * self.n - 1

    def __getitem__(self, idx):
        while True:
            rmi = random_mul_instance(self.n)        # długość 4n
            h = hash(str(rmi[:2*self.n]))            # hash zależny tylko od wejścia a+b
            inp_split = 'test' if h % 4 == 0 else 'train'
            if inp_split == self.split:
                break

        x = torch.tensor(rmi[:-1], dtype=torch.long)  # 4n-1
        y = torch.tensor(rmi[1:], dtype=torch.long)   # 4n-1

        # input length = 2n => ignorujemy y[0..2n-2] czyli 2n-1 pozycji
        y[:2*self.n - 1] = -1
        return x, y


In [5]:
train_dataset = MulDataset('train', n=3)
test_dataset  = MulDataset('test',  n=3)
x, y = train_dataset[0]

print (x)
for a, b in zip(x,y):
    print(int(a),int(b))


tensor([0, 0, 2, 9, 3, 7, 0, 0, 1, 8, 7])
0 -1
0 -1
2 -1
9 -1
3 -1
7 0
0 0
0 1
1 8
8 7
7 4


In [6]:
# create a GPT instance
from mingpt.model import GPT

model_config = GPT.get_default_config()
model_config.model_type = 'gpt-micro'
# model_config.model_type = 'gpt-nano'

model_config.vocab_size = train_dataset.get_vocab_size()
model_config.block_size = train_dataset.get_block_size()
model = GPT(model_config)

number of parameters: 0.80M


In [7]:
print (model_config.n_head, model_config.n_layer, model_config.n_embd)

4 4 128


In [8]:
# create a Trainer object
from mingpt.trainer import Trainer

train_config = Trainer.get_default_config()
train_config.learning_rate = 3e-4 # the model we're using is so small that we can go a bit faster
train_config.max_iters = 100000
train_config.num_workers = 0
trainer = Trainer(train_config, model, train_dataset)

running on device cpu


In [9]:
def batch_end_callback(trainer):
    if trainer.iter_num % 100 == 0:
        print(f"iter_dt {trainer.iter_dt * 1000:.2f}ms; iter {trainer.iter_num}: train loss {trainer.loss.item():.5f}")
trainer.set_callback('on_batch_end', batch_end_callback)

trainer.run()

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


iter_dt 0.00ms; iter 0: train loss 2.31993
iter_dt 50.91ms; iter 100: train loss 1.98773
iter_dt 48.97ms; iter 200: train loss 1.80767
iter_dt 51.79ms; iter 300: train loss 1.74184
iter_dt 53.77ms; iter 400: train loss 1.70807
iter_dt 48.01ms; iter 500: train loss 1.59851
iter_dt 51.41ms; iter 600: train loss 1.65005
iter_dt 53.16ms; iter 700: train loss 1.54870
iter_dt 55.08ms; iter 800: train loss 1.54614
iter_dt 49.33ms; iter 900: train loss 1.55102
iter_dt 49.35ms; iter 1000: train loss 1.51893
iter_dt 51.50ms; iter 1100: train loss 1.53547
iter_dt 50.69ms; iter 1200: train loss 1.49423
iter_dt 51.56ms; iter 1300: train loss 1.47758
iter_dt 50.22ms; iter 1400: train loss 1.43769
iter_dt 49.75ms; iter 1500: train loss 1.47854
iter_dt 49.43ms; iter 1600: train loss 1.46516
iter_dt 51.78ms; iter 1700: train loss 1.49992
iter_dt 48.44ms; iter 1800: train loss 1.46021
iter_dt 49.48ms; iter 1900: train loss 1.52414
iter_dt 50.61ms; iter 2000: train loss 1.42905
iter_dt 49.20ms; iter 2100

In [10]:
# now let's perform some evaluation
model.eval()
None

In [11]:
def eval_mul_split(trainer, split, max_batches=50):
    dataset = {'train': train_dataset, 'test': test_dataset}[split]
    n = dataset.n

    results = []
    loader = DataLoader(dataset, batch_size=100, num_workers=0, drop_last=False)

    for b, (x, y) in enumerate(loader):
        if b >= max_batches:
            break

        x = x.to(trainer.device)
        y = y.to(trainer.device)

        inp = x[:, :2*n]          # a(3)+b(3) => 6 tokenów
        sol = y[:, -2*n:]         # ostatnie 6 tokenów to c(6)

        cat = model.generate(inp, 2*n, do_sample=False)
        sol_candidate = cat[:, -2*n:]

        correct = (sol == sol_candidate).all(1).cpu()
        results.extend(correct.int().tolist())

    rt = torch.tensor(results, dtype=torch.float)
    print("%s final score: %d/%d = %.2f%% correct" % (split, rt.sum(), len(results), 100*rt.mean()))
    return rt.sum()


# run a lot of examples from both train and test through the model and verify the output correctness
with torch.no_grad():
    train_score = eval_mul_split(trainer, 'train', max_batches=50)
    test_score  = eval_mul_split(trainer, 'test',  max_batches=50)

train final score: 844/5000 = 16.88% correct
test final score: 799/5000 = 15.98% correct
